Feature Engineering Script for the cleaned CRMLS Sold and Listing datasets.
 
Part A -- School District Enrichment
  Downloads California school district boundaries (GeoJSON) from data.ca.gov, filters to Unified districts only, and spatially joins each property's Latitude/Longitude to find which Unified district it falls inside.
 
Part B -- Market Metric Engineering
  Adds PriceRatio, CloseToOriginalListRatio, PricePerSqFt, Year/Month/YrMo,
  ListingToContractDays, and ContractToCloseDays.
 
Part C -- Segment Analysis
  Grouped summary tables (median close price, avg price/sqft, avg DOM, etc.)
  by PropertyType x PropertySubType, CountyOrParish x MLSAreaMajor, and
  ListOfficeName x BuyerOfficeName.

METRIC DEFINITIONS

  - PriceRatio = ClosePrice / ListPrice
  - CloseToOriginalListRatio = ClosePrice / OriginalListPrice
  - PricePerSqFt = ClosePrice / LivingArea
  - DaysOnMarket = existing MLS field (not recalculated)
  - Year, Month = CloseDate.year, CloseDate.month
  - YrMo = CloseDate formatted as 'YYYY-MM' (string)
  - ListingToContractDays = PurchaseContractDate - ListingContractDate (days)
  - ContractToCloseDays = CloseDate - PurchaseContractDate (days)

MISSING / INVALID / NEGATIVE VALUE HANDLING POLICY:
  - A ratio's denominator that is missing, zero, or negative -> the ratio is set to NaN (never inf, never a nonsensical negative ratio, never a crash).
  - A date-duration whose inputs are missing (NaT) -> the duration is NaN.
  - A date-duration that comes out negative is NOT removed. It is kept and flagged (columns ending in _NegativeFlag) for a human to review.
  - DistrictName is NaN when: (a) a property's coordinates are missing/zero, (b) coordinates fall outside California, or (c) coordinates are valid and in-state but that area is served by separate Elementary + High districts rather than one Unified district (by design -- the task instructions filter to Unified only, so non-unified areas have no match).

# Install Package

In [16]:
import os
import numpy as np   # for NaN handling in the safe_divide helper
import pandas as pd   # core dataframe operations

%pip install geopandas
import geopandas as gpd  # spatial dataframes + spatial join (gpd.sjoin)

import json # for validating the downloaded GeoJSON is complete/well-formed before parsing
import requests # for downloading the district GeoJSON reliably (more robust than GDAL's URL streaming)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Config

In [3]:
DATA_DIR = r"D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv"
START_MONTH = 202401
END_MONTH = 202606
 
# Inputs: the cleaned datasets produced by clean_crmls_data.py
SOLD_INPUT_PATH = os.path.join(DATA_DIR, f"Sold_Cleaned_{START_MONTH}_{END_MONTH}.csv")
LISTING_INPUT_PATH = os.path.join(DATA_DIR, f"Listing_Cleaned_{START_MONTH}_{END_MONTH}.csv")
 
# Outputs: final feature-engineered datasets + one segment summary table
SOLD_OUTPUT_PATH = os.path.join(DATA_DIR, f"Sold_Enriched_{START_MONTH}_{END_MONTH}.csv")
LISTING_OUTPUT_PATH = os.path.join(DATA_DIR, f"Listing_Enriched_{START_MONTH}_{END_MONTH}.csv")
SEGMENT_OUTPUT_PATH = os.path.join(DATA_DIR, "Segment_Summary_County_MLSArea.csv")

In [4]:
# The direct GeoJSON download link for the CA School District Areas 2025-26
# layer (found via the data.ca.gov dataset page -> "Go to resource" for
# GeoJSON). A local cache is kept next to the CSVs so repeated runs during
# development don't re-download this large (~20-40MB) file every time.
DISTRICT_GEOJSON_URL = "https://gis.data.ca.gov/api/download/v1/items/48870daecfe14c6ab376f6a673491914/geojson?layers=0"
DISTRICT_CACHE_PATH = os.path.join(DATA_DIR, "ca_unified_school_districts_2025_26.geojson")
 
DATE_COLUMNS = ["CloseDate", "PurchaseContractDate", "ListingContractDate", "ContractStatusChangeDate"]

In [6]:
def section(title):
    """Print a visual divider in the console so each part's output is easy to scan."""
    print()
    print("=" * 78)
    print(title)
    print("=" * 78)

In [7]:
def safe_divide(numerator, denominator):
    """
    Divide numerator / denominator, but treat a denominator that is missing,
    zero, or negative as invalid. Those rows get NaN instead of inf, -inf,
    or a nonsensical negative ratio (e.g. a negative LivingArea shouldn't
    ever produce a negative price-per-sqft).
    """
    valid_denominator = denominator.mask(denominator <= 0)  # <=0 -> NaN; NaN denominators stay NaN (comparison is False)
    return numerator / valid_denominator                     # dividing by NaN naturally produces NaN, not an error

# Part A: School District Enrichment

In [17]:
section("Part A: School District Enrichment")
 
 
def load_unified_districts():
    # Reuse a cached, already-filtered copy if one exists from a previous run,
    # instead of re-downloading the full (large) source file every time.
    if os.path.exists(DISTRICT_CACHE_PATH):
        print(f"Loading cached Unified district boundaries: {DISTRICT_CACHE_PATH}")
        unified = gpd.read_file(DISTRICT_CACHE_PATH)          # read the small, pre-filtered cache
        print(f"Loaded {len(unified)} cached Unified district polygons.")
        return unified
 
    print("No cache found -- downloading CA School District boundaries from data.ca.gov...")
    print("(This file is roughly 20-40MB with ~1000 district polygons; may take a bit.)")
 
    # Download to a local file first instead of letting geopandas/GDAL stream
    # straight from the URL. GDAL's own URL streaming can silently truncate on
    # a slow or interrupted connection for a file this large, which shows up
    # as a cryptic "Unterminated string" JSON parse error. Downloading with
    # requests first, then parsing from disk, is more reliable and lets us
    # verify the download actually completed before handing it to geopandas.
    raw_download_path = os.path.join(DATA_DIR, "ca_school_districts_raw_download.geojson")
    max_attempts = 3
    for attempt in range(1, max_attempts + 1):
        try:
            print(f"  Download attempt {attempt}/{max_attempts}...")
            response = requests.get(DISTRICT_GEOJSON_URL, timeout=180)  # generous timeout for a large file
            response.raise_for_status()                                  # raise if the server returned an error status
 
            with open(raw_download_path, "wb") as f:
                f.write(response.content)                                 # write the complete response to disk
 
            # Sanity-check that what we saved is actually complete, valid
            # JSON before trusting it to geopandas -- catches truncation
            # here with a clear message instead of a confusing GDAL traceback.
            with open(raw_download_path, "r", encoding="utf-8") as f:
                json.load(f)
 
            print(f"  Downloaded and verified {os.path.getsize(raw_download_path) / 1_000_000:.1f} MB successfully.")
            break  # success -- stop retrying
 
        except (requests.exceptions.RequestException, json.JSONDecodeError) as e:
            print(f"  Attempt {attempt} failed: {e}")
            if attempt == max_attempts:
                raise SystemExit(
                    "Could not download a complete, valid copy of the school district GeoJSON "
                    f"after {max_attempts} attempts. Check your network connection, or try downloading "
                    f"the file manually from {DISTRICT_GEOJSON_URL} and saving it to:\n  {raw_download_path}\n"
                    "then re-run this script -- it will pick up the local file instead of re-downloading."
                )
 
    districts = gpd.read_file(raw_download_path)  # now parse from the verified local file, not the URL
    print(f"Downloaded {len(districts)} total district polygons (all types). CRS: {districts.crs}")
 
    # The property Latitude/Longitude values are plain WGS84 degrees, so the
    # district polygons need to be in that same coordinate system (EPSG:4326)
    # for the spatial join to line up correctly.
    if districts.crs is None:
        print("WARNING: no CRS found on the downloaded file -- assuming EPSG:4326.")
        districts = districts.set_crs(epsg=4326)                 # declare (not reproject) if CRS metadata is missing
    elif districts.crs.to_epsg() != 4326:
        print(f"Reprojecting districts from {districts.crs} to EPSG:4326 to match property coordinates.")
        districts = districts.to_crs(epsg=4326)                   # actually reproject if it's in a different CRS
 
    # Filter to Unified districts only, per instructions -- Elementary and
    # High district polygons are discarded (see the module docstring's
    # note on why some properties end up with no DistrictName as a result).
    unified = districts[districts["DistrictType"] == "Unified"].copy()
    print(f"Filtered to {len(unified)} Unified school districts (out of {len(districts)} total).")
 
    # Keep only what's needed for the join -- the source file has 40+
    # demographic columns per district that aren't relevant here.
    unified = unified[["DistrictName", "DistrictType", "geometry"]]
 
    unified.to_file(DISTRICT_CACHE_PATH, driver="GeoJSON")        # save the trimmed, filtered copy for next run
    print(f"Cached trimmed Unified-only boundaries to: {DISTRICT_CACHE_PATH}")
    return unified


Part A: School District Enrichment


In [18]:
def add_district_name(df, label, unified_districts):
    if "Latitude" not in df.columns or "Longitude" not in df.columns:  # guard: need coordinates to do anything
        print(f"- [{label}] Latitude/Longitude not found -- DistrictName set to NaN for all rows.")
        df = df.copy()
        df["DistrictName"] = np.nan
        return df
 
    # Build a Point geometry for every row from its coordinates. Geographic
    # convention is (x=longitude, y=latitude) -- NOT (latitude, longitude).
    points = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),  # one Point per row
        crs="EPSG:4326",                                                # must match the district polygons' CRS
    )
 
    # Spatial join: for each point, find which Unified district polygon
    # contains it. predicate='within' = a standard point-in-polygon test.
    # how='left' keeps every property row even when no district matches
    # (DistrictName becomes NaN for those rows rather than dropping them).
    joined = gpd.sjoin(
        points,
        unified_districts[["DistrictName", "geometry"]],
        how="left",
        predicate="within",
    )
 
    # A property could fall inside more than one Unified district polygon if
    # real-world boundary data has a slight overlap. Keep only the first
    # match per property so the row count never changes after the join.
    multi_match_count = int(joined.index.duplicated(keep="first").sum())
    if multi_match_count > 0:
        print(f"- [{label}] {multi_match_count} propert(y/ies) matched more than one Unified "
              f"district polygon (boundary overlap) -- kept the first match for each.")
    joined = joined[~joined.index.duplicated(keep="first")]
 
    # sjoin's result can be reordered -- realign back onto the original
    # dataframe's index before assigning, so each DistrictName lines up
    # with the correct row.
    df = df.copy()
    df["DistrictName"] = joined["DistrictName"].reindex(df.index)
 
    matched = int(df["DistrictName"].notna().sum())
    print(f"- [{label}] DistrictName matched for {matched}/{len(df)} rows ({matched / len(df) * 100:.1f}%)")
    return df

# Part B: Market Metric Engineering

In [10]:
def add_market_metrics(df, label, compute_close_metrics):
    df = df.copy()
 
    # ---- PriceRatio = ClosePrice / ListPrice ----
    # Only computed when ClosePrice is meaningful for this dataset (Sold)
    if compute_close_metrics and "ClosePrice" in df.columns and "ListPrice" in df.columns:
        df["PriceRatio"] = safe_divide(df["ClosePrice"], df["ListPrice"])
    else:
        print(f"- [{label}] Skipped PriceRatio (ClosePrice not used/available for this dataset).")
 
    # ---- CloseToOriginalListRatio = ClosePrice / OriginalListPrice ----
    if compute_close_metrics and "ClosePrice" in df.columns and "OriginalListPrice" in df.columns:
        df["CloseToOriginalListRatio"] = safe_divide(df["ClosePrice"], df["OriginalListPrice"])
    else:
        print(f"- [{label}] Skipped CloseToOriginalListRatio (ClosePrice not used/available).")
 
    # ---- PricePerSqFt = ClosePrice / LivingArea ----
    if compute_close_metrics and "ClosePrice" in df.columns and "LivingArea" in df.columns:
        df["PricePerSqFt"] = safe_divide(df["ClosePrice"], df["LivingArea"])
    else:
        print(f"- [{label}] Skipped PricePerSqFt (ClosePrice not used/available).")
 
    # ---- Days on Market: passthrough, nothing to compute ----
    # DaysOnMarket already exists as-is from the source MLS data
 
    # ---- Year / Month / YrMo, derived from CloseDate ----
    if compute_close_metrics and "CloseDate" in df.columns:
        df["Year"] = df["CloseDate"].dt.year.astype("Int64") # Int64 = nullable integer, keeps <NA> for missing dates
        df["Month"] = df["CloseDate"].dt.month.astype("Int64")
        yrmo_period = df["CloseDate"].dt.to_period("M").astype(str)  # e.g. '2024-06'
        df["YrMo"] = yrmo_period.where(df["CloseDate"].notna(), np.nan)  # keep truly-missing dates as NaN, not the text 'NaT'
    else:
        print(f"- [{label}] Skipped Year/Month/YrMo (CloseDate not used/available).")
 
    # ---- ListingToContractDays = PurchaseContractDate - ListingContractDate ----
    if "PurchaseContractDate" in df.columns and "ListingContractDate" in df.columns:
        df["ListingToContractDays"] = (df["PurchaseContractDate"] - df["ListingContractDate"]).dt.days  # NaT input -> NaN output
        df["ListingToContractDays_NegativeFlag"] = df["ListingToContractDays"] < 0  # NaN < 0 is False, so missing isn't flagged
    else:
        print(f"- [{label}] Skipped ListingToContractDays (required date columns not available).")
 
    # ---- ContractToCloseDays = CloseDate - PurchaseContractDate ----
    if compute_close_metrics and "CloseDate" in df.columns and "PurchaseContractDate" in df.columns:
        df["ContractToCloseDays"] = (df["CloseDate"] - df["PurchaseContractDate"]).dt.days
        df["ContractToCloseDays_NegativeFlag"] = df["ContractToCloseDays"] < 0
    else:
        print(f"- [{label}] Skipped ContractToCloseDays (CloseDate not used/available).")
 
    return df

In [11]:
def validate_metrics(df, label):
    print(f"\n--- [{label}] Metric Validation ---")
 
    # 1) Missing values in the source columns the metrics depend on
    source_cols = ["ClosePrice", "ListPrice", "OriginalListPrice", "LivingArea",
                   "PurchaseContractDate", "ListingContractDate", "CloseDate"]
    print("Missing values in source columns used by the metrics:")
    for col in source_cols:
        if col in df.columns:
            miss = int(df[col].isna().sum())
            print(f"    - {col}: {miss} missing ({miss / len(df) * 100:.1f}%)")
        else:
            print(f"    - {col}: not present in this dataset")
 
    # 2) Zero/negative denominators caught before the ratio was calculated
    print("Zero/negative denominators found (ratio set to NaN for these rows):")
    for num_col, den_col in [("ClosePrice", "ListPrice"), ("ClosePrice", "OriginalListPrice"), ("ClosePrice", "LivingArea")]:
        if den_col in df.columns:
            bad = int((df[den_col] <= 0).sum())
            print(f"    - {den_col} <= 0: {bad} rows")
 
    # 3) Negative date-duration flags (kept, not removed)
    print("Negative date-duration flags:")
    for flag_col in ["ListingToContractDays_NegativeFlag", "ContractToCloseDays_NegativeFlag"]:
        if flag_col in df.columns:
            print(f"    - {flag_col}: {int(df[flag_col].sum())} rows")
 
    # 4) Confirm the engineered columns actually have the dtypes they should
    print("Dtype confirmation for engineered columns:")
    metric_cols = ["DistrictName", "PriceRatio", "CloseToOriginalListRatio", "PricePerSqFt",
                   "Year", "Month", "YrMo", "ListingToContractDays", "ContractToCloseDays"]
    for col in metric_cols:
        if col in df.columns:
            print(f"    - {col}: {df[col].dtype}")

In [12]:
def show_sample_output(df, label):
    # Show a handful of rows where the new columns actually have values
    # so the sample demonstrably proves correct population rather than showing
    # a row that happens to be all-NaN for the newly engineered fields
    cols_of_interest = ["ListingKey", "DistrictName", "ClosePrice", "ListPrice", "OriginalListPrice",
                         "PriceRatio", "CloseToOriginalListRatio", "PricePerSqFt", "DaysOnMarket",
                         "Year", "Month", "YrMo", "ListingToContractDays", "ContractToCloseDays"]
    available_cols = [c for c in cols_of_interest if c in df.columns]
    numeric_check_cols = [c for c in available_cols if c not in ("ListingKey", "DistrictName")]
 
    complete_rows = df.dropna(subset=numeric_check_cols) if numeric_check_cols else df
    sample_source = complete_rows if len(complete_rows) > 0 else df   # fall back to any rows if none are fully complete
    sample = sample_source[available_cols].head(5)
 
    print(f"\n--- [{label}] Sample Output (newly engineered columns) ---")
    print(sample.to_string(index=False))

In [13]:
def segment_summary(df, group_cols, label):
    valid_group_cols = [c for c in group_cols if c in df.columns]
    if len(valid_group_cols) != len(group_cols):
        missing = [c for c in group_cols if c not in df.columns]
        print(f"- [{label}] Columns not found, skipping this segment: {missing}")
        return None
 
    # Only aggregate metrics that actually exist in this dataset
    agg_map = {
        "ClosePrice": ("MedianClosePrice", "median"),
        "PricePerSqFt": ("AvgPricePerSqFt", "mean"),
        "DaysOnMarket": ("AvgDaysOnMarket", "mean"),
        "PriceRatio": ("AvgPriceRatio", "mean"),
        "CloseToOriginalListRatio": ("AvgCloseToOriginalListRatio", "mean"),
        "ListingToContractDays": ("AvgListingToContractDays", "mean"),
        "ContractToCloseDays": ("AvgContractToCloseDays", "mean"),
    }
    agg_dict = {src: (new_name, how) for src, (new_name, how) in agg_map.items() if src in df.columns}
 
    grouped = df.groupby(valid_group_cols, dropna=False).agg(
        **{new_name: (src, how) for src, (new_name, how) in agg_dict.items()}
    )
    grouped["ListingCount"] = df.groupby(valid_group_cols, dropna=False).size()  # count of rows in each group
    return grouped.sort_values("ListingCount", ascending=False)

# Load Cleaned Data

In [14]:
section("Load Cleaned Data")
sold = pd.read_csv(SOLD_INPUT_PATH, low_memory=False)
listing = pd.read_csv(LISTING_INPUT_PATH, low_memory=False)
print(f"Sold: {sold.shape[0]} rows, {sold.shape[1]} columns")
print(f"Listing: {listing.shape[0]} rows, {listing.shape[1]} columns")
 
# Reading from CSV loses datetime dtype, so date columns need to be
# re-parsed here before anything in Parts A/B can use them
for col in DATE_COLUMNS:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors="coerce")
    if col in listing.columns:
        listing[col] = pd.to_datetime(listing[col], errors="coerce")


Load Cleaned Data
Sold: 447769 rows, 77 columns
Listing: 615346 rows, 68 columns


# Run Part A

In [19]:
unified_districts = load_unified_districts()
sold = add_district_name(sold, "Sold", unified_districts)
listing = add_district_name(listing, "Listing", unified_districts)

No cache found -- downloading CA School District boundaries from data.ca.gov...
(This file is roughly 20-40MB with ~1000 district polygons; may take a bit.)
  Download attempt 1/3...
  Downloaded and verified 36.7 MB successfully.
Downloaded 936 total district polygons (all types). CRS: EPSG:3857
Reprojecting districts from EPSG:3857 to EPSG:4326 to match property coordinates.
Filtered to 345 Unified school districts (out of 936 total).
Cached trimmed Unified-only boundaries to: D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv\ca_unified_school_districts_2025_26.geojson
- [Sold] DistrictName matched for 333398/447769 rows (74.5%)
- [Listing] DistrictName matched for 411540/615346 rows (66.9%)


# Run Part B

In [20]:
section("Part B: Market Metric Engineering")

# Sold: has reliable ClosePrice/CloseDate, so all 7 metrics are computed.
sold = add_market_metrics(sold, "Sold", compute_close_metrics=True)

# Listing: most rows are still-Active with no ClosePrice/CloseDate, so only
# the metrics that don't depend on a close (ListingToContractDays) are computed
# the rest are intentionally skipped rather than filled with meaningless values
listing = add_market_metrics(listing, "Listing", compute_close_metrics=False)
 
validate_metrics(sold, "Sold")
validate_metrics(listing, "Listing")
 
show_sample_output(sold, "Sold")
show_sample_output(listing, "Listing")


Part B: Market Metric Engineering
- [Listing] Skipped PriceRatio (ClosePrice not used/available for this dataset).
- [Listing] Skipped CloseToOriginalListRatio (ClosePrice not used/available).
- [Listing] Skipped PricePerSqFt (ClosePrice not used/available).
- [Listing] Skipped Year/Month/YrMo (CloseDate not used/available).
- [Listing] Skipped ContractToCloseDays (CloseDate not used/available).

--- [Sold] Metric Validation ---
Missing values in source columns used by the metrics:
    - ClosePrice: 2 missing (0.0%)
    - ListPrice: 0 missing (0.0%)
    - OriginalListPrice: 822 missing (0.2%)
    - LivingArea: 253 missing (0.1%)
    - PurchaseContractDate: 198 missing (0.0%)
    - ListingContractDate: 1 missing (0.0%)
    - CloseDate: 0 missing (0.0%)
Zero/negative denominators found (ratio set to NaN for these rows):
    - ListPrice <= 0: 0 rows
    - OriginalListPrice <= 0: 2 rows
    - LivingArea <= 0: 0 rows
Negative date-duration flags:
    - ListingToContractDays_NegativeFlag: 2

# Run Part C: Segment Analysis (Sold dataset)

In [21]:
section("Part C: Segment Analysis (Sold dataset)")

segments_to_run = {
    "PropertyType x PropertySubType": ["PropertyType", "PropertySubType"],
    "CountyOrParish x MLSAreaMajor": ["CountyOrParish", "MLSAreaMajor"],
    "ListOfficeName x BuyerOfficeName": ["ListOfficeName", "BuyerOfficeName"],
}
 
segment_tables = {}
for seg_label, seg_cols in segments_to_run.items():
    table = segment_summary(sold, seg_cols, seg_label)
    segment_tables[seg_label] = table
    if table is not None:
        print(f"\n--- {seg_label} (top 10 by listing count) ---")
        print(table.head(10).to_string())


Part C: Segment Analysis (Sold dataset)

--- PropertyType x PropertySubType (top 10 by listing count) ---
                                    MedianClosePrice  AvgPricePerSqFt  AvgDaysOnMarket  AvgPriceRatio  AvgCloseToOriginalListRatio  AvgListingToContractDays  AvgContractToCloseDays  ListingCount
PropertyType PropertySubType                                                                                                                                                                      
Residential  SingleFamilyResidence          895000.0       637.318573        36.102210       1.061084                    56.772791                 44.190208               31.506132        335426
             Condominium                    627000.0       704.227307        41.935586       1.172134                    24.057885                 48.778447               30.873091         73587
             Townhouse                      800000.0       656.989357        32.277401       1.125475            

# Save Outputs

In [22]:
section("Save Outputs")
sold.to_csv(SOLD_OUTPUT_PATH, index=False)
listing.to_csv(LISTING_OUTPUT_PATH, index=False)
print(f"Saved: {SOLD_OUTPUT_PATH}")
print(f"Saved: {LISTING_OUTPUT_PATH}")
 
county_area_table = segment_tables.get("CountyOrParish x MLSAreaMajor")
if county_area_table is not None:
    county_area_table.to_csv(SEGMENT_OUTPUT_PATH, index=True)
    print(f"Saved: {SEGMENT_OUTPUT_PATH}")


Save Outputs
Saved: D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv\Sold_Enriched_202401_202606.csv
Saved: D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv\Listing_Enriched_202401_202606.csv
Saved: D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv\Segment_Summary_County_MLSArea.csv
